<a href="https://colab.research.google.com/github/anjali595/Autonomous-ML-Model-Monitoring-Drift-Management-Platform/blob/main/autonomous-drift-Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GOAL** :- To build an platform which continuously monitors model inputs and predictions, detects data drift using statistical tests, and autonomously retrains models when performance degradation occurs.

**Dataset used**:- **LOAN PREDICTION PROBLEM DATASET** (Kaggle)


Downloading useful libraries

In [1]:
!pip install evidently
!pip install scikit-learn
!pip install pandas
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.8/237.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.6/567.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.4 MB/s eta 0:00:00


**Importing libraries**

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

import joblib

In [3]:
train = pd.read_csv("/content/train_u6lujuX_CVtuZ9i.csv")
test = pd.read_csv("/content/test_Y3wMUE5_7gLdaTN.csv")

print(train.head())

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0   

**Data Preprocessing**

In [4]:
train = train.drop("Loan_ID", axis=1)

# Fill missing values
for col in train.columns:
    if train[col].dtype == "object":
        train[col].fillna(train[col].mode()[0], inplace=True)
    else:
        train[col].fillna(train[col].median(), inplace=True)

# Encode categorical variables
le = LabelEncoder()

for col in train.columns:
    if train[col].dtype == "object":
        train[col] = le.fit_transform(train[col])

/tmp/ipykernel_332/723561195.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].mode()[0], inplace=True)
/tmp/ipykernel_332/723561195.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

**Data Splitting using train-test-split**

In [5]:
X = train.drop("Loan_Status", axis=1)
y = train["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Train the ML Model**

In [6]:
model = RandomForestClassifier()

model.fit(X_train, y_train)

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.7804878048780488


**Saving the model/pickel file generated**

In [7]:
joblib.dump(model, "loan_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


**Build Prediction API Simulation**

This acts like a production API.

In [8]:
def predict_loan(input_data):

    model = joblib.load("loan_model.pkl")

    input_df = pd.DataFrame([input_data])

    prediction = model.predict(input_df)[0]

    return prediction

testing above

In [9]:
sample = X_test.iloc[0].to_dict()

predict_loan(sample)

np.int64(1)

**Prediction Logging System**

In [10]:
prediction_logs = []

def log_prediction(features, prediction):

    log = {
        "features": features,
        "prediction": prediction
    }

    prediction_logs.append(log)

**Simulate Production Predictions**

In [11]:
for i in range(50):

    sample = X_test.sample(1)

    pred = model.predict(sample)[0]

    log_prediction(sample.to_dict(), pred)

**Drift Detection Engine**

We detect data drift using KS Test.

In [12]:
from scipy.stats import ks_2samp

def detect_drift(train_data, production_data):

    drift_report = {}

    for column in train_data.columns:

        stat, p_value = ks_2samp(train_data[column], production_data[column])

        drift_report[column] = p_value

    return drift_report

**Simulate Production Data**

If p-value < 0.05 → Drift Detected

In [13]:
production_data = X_test.sample(100)

drift_results = detect_drift(X_train, production_data)

print(drift_results)

{'Gender': np.float64(0.7877808143616286), 'Married': np.float64(0.29920728765662274), 'Dependents': np.float64(0.999999848579403), 'Education': np.float64(1.0), 'Self_Employed': np.float64(1.0), 'ApplicantIncome': np.float64(0.04914026225242494), 'CoapplicantIncome': np.float64(0.9204227871071752), 'LoanAmount': np.float64(0.5385995191386056), 'Loan_Amount_Term': np.float64(0.9419664062306309), 'Credit_History': np.float64(1.0), 'Property_Area': np.float64(0.9901910506466424)}


**Model Performance Monitoring**

In [14]:
def monitor_model():

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)

    print("Current Accuracy:", acc)

    if acc < 0.75:
        print("⚠️ Performance Drop Detected")

**Automated Retraining** **Pipeline**

In [15]:
def retrain_model():

    print("Retraining Model...")

    new_model = RandomForestClassifier()

    new_model.fit(X_train, y_train)

    joblib.dump(new_model, "loan_model.pkl")

    print("Model Retrained and Updated")

**Autonomous Action Agent**

basically the major requirement of our project (Brain of our system)

In [16]:
def action_agent():

    drift = detect_drift(X_train, X_test)

    for feature, p in drift.items():

        if p < 0.05:

            print("Drift detected in:", feature)

            retrain_model()

            break

**Testing**

In [17]:
monitor_model()

action_agent()

Current Accuracy: 0.7804878048780488
Drift detected in: ApplicantIncome
Retraining Model...
Model Retrained and Updated
